# Distributed Rate Limiter Design — Interview Notes

# 1. What is a Rate Limiter?

A rate limiter controls how many requests a client can make in a given time window.

Example:

```text
100 requests per minute per user
```

If limit exceeds:

```http
429 Too Many Requests
```

---

# 2. Why Do We Need It?

Prevents:
- DDoS attacks
- API abuse
- Brute force login attempts
- Server overload
- Excessive resource usage

---

# 3. Functional Requirements

- Limit requests per user/IP/API key
- Configurable limits
- Low latency
- Distributed support
- High availability

---

# 4. High-Level Design

```text
Client
   ↓
Load Balancer
   ↓
API Gateway / Rate Limiter
   ↓
Application Servers
   ↓
Database
```

---

# 5. Key Design Components

| Component | Purpose |
|---|---|
| Identifier | User ID / IP / API Key |
| Counter | Counts requests |
| Time Window | 1 sec / 1 min |
| Storage | Redis |
| Decision Engine | Allow/Reject |

---

# 6. Algorithm Comparison

## Fixed Window Counter

### Problem

```text
100 req at 12:00:59
+
100 req at 12:01:00
```

Effectively:

```text
200 requests in 2 sec
```

---

## Sliding Window Log

### Pros
- Accurate
- Smooth limiting

### Cons
- High memory usage
- Expensive at scale

---

## Token Bucket (Recommended)

### Example

```text
Bucket Size = 10
Refill Rate = 1 token/sec
```

Each request consumes one token.

---

# 7. Why Token Bucket?

| Feature | Benefit |
|---|---|
| Allows bursts | Better UX |
| Smooth traffic | Avoids spikes |
| Memory efficient | Scalable |
| Simple | Easy implementation |

---

# 8. Why Redis?

- Fast
- In-memory
- Atomic operations
- TTL support
- Distributed

---

# 9. Redis Key Design

```text
rate_limit:user123
```

---

# 10. Request Flow

```text
1. Request arrives
2. Extract user ID/API key
3. Build Redis key
4. Fetch current tokens
5. Refill tokens if needed
6. Check availability
7. If token exists:
      decrement token
      allow request
8. Else:
      return HTTP 429
```

---

# 11. Pseudocode

```python
def allow_request(user_id):

    tokens = redis.get(user_id)

    if tokens > 0:
        redis.decr(user_id)
        return True

    return False
```

---

# 12. Distributed Architecture

```text
Clients
   ↓
Load Balancer
   ↓
API Servers
   ↓
Shared Redis Cluster
```

---

# 13. Concurrency Problem

Two requests may read:

```text
tokens = 1
```

Both may allow request incorrectly.

## Solution

- Atomic Redis operations
- Lua scripts

---

# 14. Failure Handling

## Fail Open

Allow traffic temporarily.

## Fail Closed

Reject requests for safety.

---

# 15. Monitoring

Track:
- Rejected requests
- Redis latency
- Traffic spikes
- Abusive IPs

Tools:
- Prometheus
- Grafana

---

# 16. Final Production Architecture

```text
Client
   ↓
CDN/WAF
   ↓
Load Balancer
   ↓
API Gateway
   ↓
Redis-based Token Bucket Limiter
   ↓
Microservices
```

---

# 17. Final Interview Summary

Use a Redis-backed Token Bucket rate limiter because it provides:
- Low latency
- Burst handling
- Scalability
- Efficient distributed coordination

---

# 18. Golden Interview Flow

```text
Requirements
→ High-level design
→ Algorithm comparison
→ Chosen approach
→ Distributed concerns
→ Failure handling
→ Scaling
→ Monitoring
```
